In [24]:
import requests
import os
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed


download_uris = [
    "https://divvy-tripdata.s3.amazonaws.com/Divvy_Trips_2018_Q4.zip",
    "https://divvy-tripdata.s3.amazonaws.com/Divvy_Trips_2019_Q1.zip",
    "https://divvy-tripdata.s3.amazonaws.com/Divvy_Trips_2019_Q2.zip",
    "https://divvy-tripdata.s3.amazonaws.com/Divvy_Trips_2220_Q1.zip",
]

os.makedirs("downloads", exist_ok=True)

def download_file(url):
    file_name = url.split("/")[-1]
    print(f"Starting: {file_name}")

    try: 
        response = requests.get(url)
        
        response.raise_for_status()

        with open(f"downloads/{file_name}", "wb") as file:
            file.write(response.content)

        with zipfile.ZipFile(f"downloads/{file_name}", "r") as zip_file:
            zip_file.extractall("downloads")

        os.remove(f"downloads/{file_name}")

        print(f"Successfully downloaded:{file_name}")

        return file_name

    except requests.exceptions.RequestException as error:
        print(f"Failed to download {file_name}: {error}")

futures = []

with ThreadPoolExecutor(max_workers=3) as executor:
    for url in download_uris:
        future = executor.submit(download_file, url)
        futures.append(future)

    for future in as_completed(futures):
        result = future.result()
        print(f"Result: {result}")

Starting: Divvy_Trips_2018_Q4.zip
Starting: Divvy_Trips_2019_Q1.zip
Starting: Divvy_Trips_2019_Q2.zip
Successfully downloaded:Divvy_Trips_2019_Q1.zip
Starting: Divvy_Trips_2220_Q1.zip
Result: Divvy_Trips_2019_Q1.zip
Failed to download Divvy_Trips_2220_Q1.zip: 404 Client Error: Not Found for url: https://divvy-tripdata.s3.amazonaws.com/Divvy_Trips_2220_Q1.zip
Result: None
Successfully downloaded:Divvy_Trips_2018_Q4.zip
Result: Divvy_Trips_2018_Q4.zip
Successfully downloaded:Divvy_Trips_2019_Q2.zip
Result: Divvy_Trips_2019_Q2.zip
